In [2]:
# ================================================================
# PHASE 3: AIRCRAFT ROTATION RECONSTRUCTION
# ================================================================

import pandas as pd
import numpy as np
import os
import airportsdata

from IPython.display import display

project_path = r"C:\Users\bhavy\flight-delay-domino-effect"

processed_path = os.path.join(
    project_path, "data", "processed"
)

rotation_flights = pd.read_csv(
    os.path.join(processed_path, "rotation_flights.csv")
)

phase3_flights = rotation_flights.copy()

phase3_flights["FL_DATE"] = pd.to_datetime(
    phase3_flights["FL_DATE"], errors="coerce"
).dt.normalize()

required_columns = [
    "FL_DATE", "TAIL_NUM", "OP_UNIQUE_CARRIER",
    "OP_CARRIER_FL_NUM", "ORIGIN", "DEST",
    "CRS_DEP_TIME", "DEP_TIME",
    "CRS_ARR_TIME", "ARR_TIME",
    "DEP_DELAY", "ARR_DELAY",
    "DEP_UTC", "ARR_UTC"
]

missing_columns = [
    col for col in required_columns
    if col not in phase3_flights.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# Reuse the actual UTC timestamps validated during cleaning
phase3_flights["DEP_UTC"] = pd.to_datetime(
    phase3_flights["DEP_UTC"], errors="coerce", utc=True
)

phase3_flights["ARR_UTC"] = pd.to_datetime(
    phase3_flights["ARR_UTC"], errors="coerce", utc=True
)

# Normalize identifiers
for col in ["ORIGIN", "DEST"]:
    phase3_flights[col] = (
        phase3_flights[col]
        .astype("string")
        .str.strip()
        .str.upper()
    )

phase3_flights["TAIL_NUM"] = (
    phase3_flights["TAIL_NUM"]
    .astype("string")
    .str.strip()
)

print("PHASE 3 INPUT VALIDATION")
print("-" * 50)
print("Loaded flights:", len(phase3_flights))
print("Columns:", len(phase3_flights.columns))
print("Unique aircraft:", phase3_flights["TAIL_NUM"].nunique())

print(
    "Missing departure UTC:",
    phase3_flights["DEP_UTC"].isna().sum()
)

print(
    "Missing arrival UTC:",
    phase3_flights["ARR_UTC"].isna().sum()
)

display(phase3_flights.head())

PHASE 3 INPUT VALIDATION
--------------------------------------------------
Loaded flights: 517222
Columns: 33
Unique aircraft: 5698
Missing departure UTC: 0
Missing arrival UTC: 0


,MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,DEP_TIME,...,LATE_AIRCRAFT_DELAY,DEP_TIME_TD,ARR_TIME_TD,CRS_DEP_TIME_TD,CRS_ARR_TIME_TD,DEP_UTC,ARR_UTC,FLIGHT_DURATION,DEP_LOCAL,ARR_LOCAL
0,1,4,2026-01-01,G4,188NV,1397,TYS,BOS,945,934.0,...,NaN,0 days 09:34:00,0 days 11:58:00,0 days 09:45:00,0 days 11:55:00,2026-01-01 14:34:00+00:00,2026-01-01 16:58:00+00:00,0 days 02:24:00,2026-01-01 09:34:00,2026-01-01 11:58:00
1,1,4,2026-01-01,G4,188NV,1398,BOS,TYS,1245,1240.0,...,NaN,0 days 12:40:00,0 days 15:07:00,0 days 12:45:00,0 days 15:16:00,2026-01-01 17:40:00+00:00,2026-01-01 20:07:00+00:00,0 days 02:27:00,2026-01-01 12:40:00,2026-01-01 15:07:00
2,1,5,2026-01-02,G4,188NV,1396,TYS,EWR,715,706.0,...,NaN,0 days 07:06:00,0 days 09:00:00,0 days 07:15:00,0 days 09:05:00,2026-01-02 12:06:00+00:00,2026-01-02 14:00:00+00:00,0 days 01:54:00,2026-01-02 07:06:00,2026-01-02 09:00:00
3,1,5,2026-01-02,G4,188NV,1399,EWR,TYS,955,947.0,...,NaN,0 days 09:47:00,0 days 11:48:00,0 days 09:55:00,0 days 11:55:00,2026-01-02 14:47:00+00:00,2026-01-02 16:48:00+00:00,0 days 02:01:00,2026-01-02 09:47:00,2026-01-02 11:48:00
4,1,5,2026-01-02,G4,188NV,3030,TYS,PIE,1245,1239.0,...,NaN,0 days 12:39:00,0 days 14:22:00,0 days 12:45:00,0 days 14:27:00,2026-01-02 17:39:00+00:00,2026-01-02 19:22:00+00:00,0 days 01:43:00,2026-01-02 12:39:00,2026-01-02 14:22:00


In [3]:
# ================================================================
# 2. AIRPORT TIMEZONE MAPPING
# ================================================================

airports = airportsdata.load("IATA")

airport_codes = (
    set(phase3_flights["ORIGIN"].dropna()) |
    set(phase3_flights["DEST"].dropna())
)

airport_tz = {
    code: airports[code]["tz"]
    for code in airport_codes
    if code in airports
}

# Retain the manual correction used in our earlier cleaning
if "PBI" in airport_codes:
    airport_tz["PBI"] = "America/New_York"

missing_airports = airport_codes - set(airport_tz)

print("Total airports:", len(airport_codes))
print("Mapped airports:", len(airport_tz))
print("Missing airports:", missing_airports)

if missing_airports:
    raise ValueError(
        f"Missing timezone mappings: {missing_airports}"
    )


# ================================================================
# 3. CONVERT HHMM VALUES INTO TIME DURATIONS
# ================================================================

def convert_hhmm_series(series):

    values = pd.to_numeric(series, errors="coerce")

    hours = np.floor(values / 100)
    minutes = values % 100

    valid = (
        values.notna() &
        (values % 1 == 0) &
        (hours >= 0) &
        (hours <= 24) &
        (minutes >= 0) &
        (minutes < 60) &
        ((hours < 24) | (minutes == 0))
    )

    total_minutes = hours * 60 + minutes

    return pd.to_timedelta(
        total_minutes.where(valid),
        unit="m"
    )


for col in ["CRS_DEP_TIME", "CRS_ARR_TIME"]:
    phase3_flights[col + "_TD"] = convert_hhmm_series(
        phase3_flights[col]
    )


# ================================================================
# 4. CREATE SCHEDULED DEPARTURE UTC
# ================================================================

phase3_flights["CRS_DEP_LOCAL"] = (
    phase3_flights["FL_DATE"] +
    phase3_flights["CRS_DEP_TIME_TD"]
)


def local_to_utc(local_times, airport_codes, timezone_map):

    result = pd.Series(
        pd.NaT,
        index=local_times.index,
        dtype="datetime64[ns, UTC]"
    )

    for airport in airport_codes.dropna().unique():

        mask = airport_codes == airport
        timezone = timezone_map[airport]

        localized = local_times.loc[mask].dt.tz_localize(
            timezone,
            ambiguous="NaT",
            nonexistent="shift_forward"
        )

        result.loc[mask] = localized.dt.tz_convert("UTC")

    return result


phase3_flights["CRS_DEP_UTC"] = local_to_utc(
    phase3_flights["CRS_DEP_LOCAL"],
    phase3_flights["ORIGIN"],
    airport_tz
)


# ================================================================
# 5. CREATE SCHEDULED ARRIVAL UTC
#    Handles overnight flights and date-line crossings
# ================================================================

def create_scheduled_arrival_utc(df):

    result = pd.Series(
        pd.NaT,
        index=df.index,
        dtype="datetime64[ns, UTC]"
    )

    for airport in df["DEST"].dropna().unique():

        mask = df["DEST"] == airport
        timezone = airport_tz[airport]

        base_local = (
            df.loc[mask, "FL_DATE"] +
            df.loc[mask, "CRS_ARR_TIME_TD"]
        )

        departure_utc = df.loc[mask, "CRS_DEP_UTC"]

        best_arrival = pd.Series(
            pd.NaT,
            index=base_local.index,
            dtype="datetime64[ns, UTC]"
        )

        best_gap = pd.Series(
            np.inf,
            index=base_local.index
        )

        for day_offset in [-1, 0, 1]:

            candidate_local = (
                base_local +
                pd.Timedelta(days=day_offset)
            )

            candidate_utc = (
                candidate_local
                .dt.tz_localize(
                    timezone,
                    ambiguous="NaT",
                    nonexistent="shift_forward"
                )
                .dt.tz_convert("UTC")
            )

            gap = (
                candidate_utc - departure_utc
            ).dt.total_seconds() / 60

            valid = (
                candidate_utc.notna() &
                departure_utc.notna() &
                (gap > 0) &
                (gap <= 1440) &
                (gap < best_gap)
            )

            best_arrival.loc[valid] = (
                candidate_utc.loc[valid]
            )

            best_gap.loc[valid] = gap.loc[valid]

        result.loc[mask] = best_arrival

    return result


phase3_flights["CRS_ARR_UTC"] = (
    create_scheduled_arrival_utc(phase3_flights)
)


print("Scheduled timestamp creation completed.")

print(
    "Missing scheduled departure UTC:",
    phase3_flights["CRS_DEP_UTC"].isna().sum()
)

print(
    "Missing scheduled arrival UTC:",
    phase3_flights["CRS_ARR_UTC"].isna().sum()
)

Total airports: 341
Mapped airports: 341
Missing airports: set()
Scheduled timestamp creation completed.
Missing scheduled departure UTC: 0
Missing scheduled arrival UTC: 0


In [4]:
# ================================================================
# 6. FLIGHT DURATION CALCULATION
# ================================================================

phase3_flights["FLIGHT_DURATION_MIN"] = (
    (
        phase3_flights["ARR_UTC"] -
        phase3_flights["DEP_UTC"]
    ).dt.total_seconds() / 60
)

phase3_flights["SCHEDULED_FLIGHT_DURATION_MIN"] = (
    (
        phase3_flights["CRS_ARR_UTC"] -
        phase3_flights["CRS_DEP_UTC"]
    ).dt.total_seconds() / 60
)


# Identify invalid actual flight timestamps

invalid_timestamp_mask = (
    phase3_flights["DEP_UTC"].isna() |
    phase3_flights["ARR_UTC"].isna() |
    (phase3_flights["FLIGHT_DURATION_MIN"] <= 0) |
    (phase3_flights["FLIGHT_DURATION_MIN"] > 1440)
)

invalid_flights = phase3_flights[
    invalid_timestamp_mask
].copy()

phase3_flights = phase3_flights[
    ~invalid_timestamp_mask
].copy()

print("TIMESTAMP VALIDATION")
print("-" * 50)
print("Invalid flights:", len(invalid_flights))
print("Valid flights:", len(phase3_flights))

TIMESTAMP VALIDATION
--------------------------------------------------
Invalid flights: 0
Valid flights: 517222


In [5]:
# ================================================================
# 7. AIRCRAFT GROUPING AND CHRONOLOGICAL SORTING
# ================================================================

phase3_flights = phase3_flights.dropna(
    subset=["TAIL_NUM", "DEP_UTC", "ARR_UTC"]
).copy()

phase3_flights = phase3_flights.sort_values(
    by=["TAIL_NUM", "DEP_UTC", "ARR_UTC"]
).reset_index(drop=True)

phase3_flights["FLIGHT_ID"] = (
    "FL" + phase3_flights.index.astype(str)
)

print("Aircraft grouping completed.")
print("Valid flights:", len(phase3_flights))
print("Unique aircraft:", phase3_flights["TAIL_NUM"].nunique())


# ================================================================
# 8. IDENTIFY CONSECUTIVE FLIGHTS
# ================================================================

next_columns = [
    "FLIGHT_ID",
    "TAIL_NUM",
    "OP_UNIQUE_CARRIER",
    "OP_CARRIER_FL_NUM",
    "FL_DATE",
    "ORIGIN",
    "DEST",
    "DEP_UTC",
    "ARR_UTC",
    "CRS_DEP_UTC",
    "CRS_ARR_UTC",
    "DEP_DELAY",
    "ARR_DELAY"
]

grouped = phase3_flights.groupby(
    "TAIL_NUM",
    sort=False
)

for col in next_columns:

    phase3_flights["NEXT_" + col] = (
        grouped[col].shift(-1)
    )

candidate_connections = phase3_flights[
    phase3_flights["NEXT_FLIGHT_ID"].notna()
].copy()

print("Candidate connections:", len(candidate_connections))

Aircraft grouping completed.
Valid flights: 517222
Unique aircraft: 5698
Candidate connections: 511524


In [6]:
# ================================================================
# 9. CREATE CONNECTION DATAFRAME
# ================================================================

connection_mapping = {
    "PREV_FLIGHT_ID": "FLIGHT_ID",
    "PREV_TAIL_NUM": "TAIL_NUM",
    "PREV_CARRIER": "OP_UNIQUE_CARRIER",
    "PREV_FLIGHT_NUM": "OP_CARRIER_FL_NUM",
    "PREV_FL_DATE": "FL_DATE",
    "PREV_ORIGIN": "ORIGIN",
    "PREV_DEST": "DEST",
    "PREV_DEP_UTC": "DEP_UTC",
    "PREV_ARR_UTC": "ARR_UTC",
    "PREV_CRS_DEP_UTC": "CRS_DEP_UTC",
    "PREV_CRS_ARR_UTC": "CRS_ARR_UTC",
    "PREV_DEP_DELAY": "DEP_DELAY",
    "PREV_ARR_DELAY": "ARR_DELAY",

    "NEXT_FLIGHT_ID": "NEXT_FLIGHT_ID",
    "NEXT_TAIL_NUM": "NEXT_TAIL_NUM",
    "NEXT_CARRIER": "NEXT_OP_UNIQUE_CARRIER",
    "NEXT_FLIGHT_NUM": "NEXT_OP_CARRIER_FL_NUM",
    "NEXT_FL_DATE": "NEXT_FL_DATE",
    "NEXT_ORIGIN": "NEXT_ORIGIN",
    "NEXT_DEST": "NEXT_DEST",
    "NEXT_DEP_UTC": "NEXT_DEP_UTC",
    "NEXT_ARR_UTC": "NEXT_ARR_UTC",
    "NEXT_CRS_DEP_UTC": "NEXT_CRS_DEP_UTC",
    "NEXT_CRS_ARR_UTC": "NEXT_CRS_ARR_UTC",
    "NEXT_DEP_DELAY": "NEXT_DEP_DELAY",
    "NEXT_ARR_DELAY": "NEXT_ARR_DELAY"
}

aircraft_connections = pd.DataFrame(
    index=candidate_connections.index
)

for new_col, old_col in connection_mapping.items():

    aircraft_connections[new_col] = (
        candidate_connections[old_col]
    )

aircraft_connections = aircraft_connections.reset_index(drop=True)

print("Connection dataframe created.")
print("Candidate connections:", len(aircraft_connections))

display(aircraft_connections.head())

Connection dataframe created.
Candidate connections: 511524


,PREV_FLIGHT_ID,PREV_TAIL_NUM,PREV_CARRIER,PREV_FLIGHT_NUM,PREV_FL_DATE,PREV_ORIGIN,PREV_DEST,PREV_DEP_UTC,PREV_ARR_UTC,PREV_CRS_DEP_UTC,...,NEXT_FLIGHT_NUM,NEXT_FL_DATE,NEXT_ORIGIN,NEXT_DEST,NEXT_DEP_UTC,NEXT_ARR_UTC,NEXT_CRS_DEP_UTC,NEXT_CRS_ARR_UTC,NEXT_DEP_DELAY,NEXT_ARR_DELAY
0,FL0,188NV,G4,1397,2026-01-01,TYS,BOS,2026-01-01 14:34:00+00:00,2026-01-01 16:58:00+00:00,2026-01-01 14:45:00+00:00,...,1398.0,2026-01-01,BOS,TYS,2026-01-01 17:40:00+00:00,2026-01-01 20:07:00+00:00,2026-01-01 17:45:00+00:00,2026-01-01 20:16:00+00:00,-5.0,-9.0
1,FL1,188NV,G4,1398,2026-01-01,BOS,TYS,2026-01-01 17:40:00+00:00,2026-01-01 20:07:00+00:00,2026-01-01 17:45:00+00:00,...,1396.0,2026-01-02,TYS,EWR,2026-01-02 12:06:00+00:00,2026-01-02 14:00:00+00:00,2026-01-02 12:15:00+00:00,2026-01-02 14:05:00+00:00,-9.0,-5.0
2,FL2,188NV,G4,1396,2026-01-02,TYS,EWR,2026-01-02 12:06:00+00:00,2026-01-02 14:00:00+00:00,2026-01-02 12:15:00+00:00,...,1399.0,2026-01-02,EWR,TYS,2026-01-02 14:47:00+00:00,2026-01-02 16:48:00+00:00,2026-01-02 14:55:00+00:00,2026-01-02 16:55:00+00:00,-8.0,-7.0
3,FL3,188NV,G4,1399,2026-01-02,EWR,TYS,2026-01-02 14:47:00+00:00,2026-01-02 16:48:00+00:00,2026-01-02 14:55:00+00:00,...,3030.0,2026-01-02,TYS,PIE,2026-01-02 17:39:00+00:00,2026-01-02 19:22:00+00:00,2026-01-02 17:45:00+00:00,2026-01-02 19:27:00+00:00,-6.0,-5.0
4,FL4,188NV,G4,3030,2026-01-02,TYS,PIE,2026-01-02 17:39:00+00:00,2026-01-02 19:22:00+00:00,2026-01-02 17:45:00+00:00,...,1397.0,2026-01-02,PIE,TYS,2026-01-02 20:06:00+00:00,2026-01-02 21:53:00+00:00,2026-01-02 20:17:00+00:00,2026-01-02 21:56:00+00:00,-11.0,-3.0


In [7]:
# ================================================================
# 10. AIRCRAFT AND AIRPORT CONTINUITY
# ================================================================

aircraft_connections["SAME_AIRCRAFT"] = (
    aircraft_connections["PREV_TAIL_NUM"] ==
    aircraft_connections["NEXT_TAIL_NUM"]
)

aircraft_connections["AIRPORT_CONTINUITY"] = (
    aircraft_connections["PREV_DEST"] ==
    aircraft_connections["NEXT_ORIGIN"]
)

aircraft_connections["ACTUAL_TIME_VALID"] = (
    aircraft_connections["NEXT_DEP_UTC"] >
    aircraft_connections["PREV_ARR_UTC"]
)


# ================================================================
# 11. TURNAROUND CALCULATION
# ================================================================

aircraft_connections["ACTUAL_TURNAROUND_MIN"] = (
    (
        aircraft_connections["NEXT_DEP_UTC"] -
        aircraft_connections["PREV_ARR_UTC"]
    ).dt.total_seconds() / 60
)

aircraft_connections["SCHEDULED_TURNAROUND_BUFFER_MIN"] = (
    (
        aircraft_connections["NEXT_CRS_DEP_UTC"] -
        aircraft_connections["PREV_CRS_ARR_UTC"]
    ).dt.total_seconds() / 60
)

aircraft_connections["BUFFER_CONSUMED_MIN"] = (
    aircraft_connections["SCHEDULED_TURNAROUND_BUFFER_MIN"] -
    aircraft_connections["ACTUAL_TURNAROUND_MIN"]
)

aircraft_connections["SHORT_TURNAROUND_FLAG"] = (
    aircraft_connections["ACTUAL_TURNAROUND_MIN"] < 20
)

print("Turnaround calculations completed.")

display(
    aircraft_connections[
        [
            "PREV_TAIL_NUM",
            "PREV_DEST",
            "NEXT_ORIGIN",
            "ACTUAL_TURNAROUND_MIN",
            "SCHEDULED_TURNAROUND_BUFFER_MIN",
            "BUFFER_CONSUMED_MIN"
        ]
    ].head(10)
)

Turnaround calculations completed.


,PREV_TAIL_NUM,PREV_DEST,NEXT_ORIGIN,ACTUAL_TURNAROUND_MIN,SCHEDULED_TURNAROUND_BUFFER_MIN,BUFFER_CONSUMED_MIN
0,188NV,BOS,BOS,42.0,50.0,8.0
1,188NV,TYS,TYS,959.0,959.0,0.0
2,188NV,EWR,EWR,47.0,50.0,3.0
3,188NV,TYS,TYS,51.0,50.0,-1.0
4,188NV,PIE,PIE,44.0,50.0,6.0
5,188NV,TYS,TYS,55.0,50.0,-5.0
6,188NV,SFB,SFB,44.0,50.0,6.0
7,188NV,TYS,TYS,475.0,492.0,17.0
8,188NV,BOS,BOS,63.0,50.0,-13.0
9,188NV,TYS,TYS,62.0,50.0,-12.0


In [8]:
# ================================================================
# 12. CONNECTION VALIDATION
# ================================================================

actual_turnaround_valid = (
    (aircraft_connections["ACTUAL_TURNAROUND_MIN"] > 0) &
    (aircraft_connections["ACTUAL_TURNAROUND_MIN"] <= 1440)
)

scheduled_turnaround_valid = (
    (aircraft_connections["SCHEDULED_TURNAROUND_BUFFER_MIN"] > 0) &
    (aircraft_connections["SCHEDULED_TURNAROUND_BUFFER_MIN"] <= 1440)
)

scheduled_timestamps_valid = (
    aircraft_connections["PREV_CRS_ARR_UTC"].notna() &
    aircraft_connections["NEXT_CRS_DEP_UTC"].notna()
)

checks = {
    "DIFFERENT_AIRCRAFT":
        ~aircraft_connections["SAME_AIRCRAFT"],

    "AIRPORT_MISMATCH":
        ~aircraft_connections["AIRPORT_CONTINUITY"],

    "INVALID_ACTUAL_TIME":
        ~aircraft_connections["ACTUAL_TIME_VALID"],

    "INVALID_ACTUAL_TURNAROUND":
        ~actual_turnaround_valid,

    "MISSING_SCHEDULED_TIMESTAMPS":
        ~scheduled_timestamps_valid,

    "INVALID_SCHEDULED_TURNAROUND":
        ~scheduled_turnaround_valid
}

reasons = pd.Series(
    "",
    index=aircraft_connections.index,
    dtype="object"
)

for reason, condition in checks.items():

    mask = condition.fillna(True)

    reasons.loc[mask] = reasons.loc[mask].apply(
        lambda current:
            current + "; " + reason
            if current else reason
    )

aircraft_connections["CONNECTION_VALID"] = (
    aircraft_connections["SAME_AIRCRAFT"] &
    aircraft_connections["AIRPORT_CONTINUITY"] &
    aircraft_connections["ACTUAL_TIME_VALID"] &
    actual_turnaround_valid &
    scheduled_timestamps_valid &
    scheduled_turnaround_valid
)

aircraft_connections["CONNECTION_REJECTION_REASON"] = (
    reasons.replace("", "VALID")
)

candidate_connections = aircraft_connections.copy()

rejected_connections = aircraft_connections[
    ~aircraft_connections["CONNECTION_VALID"]
].copy()

aircraft_connections = aircraft_connections[
    aircraft_connections["CONNECTION_VALID"]
].copy().reset_index(drop=True)

rejected_connections = rejected_connections.reset_index(drop=True)

print("CONNECTION VALIDATION RESULTS")
print("-" * 50)
print("Candidate connections:", len(candidate_connections))
print("Valid connections:", len(aircraft_connections))
print("Rejected connections:", len(rejected_connections))

print(
    "Valid connection percentage:",
    round(
        len(aircraft_connections) /
        max(len(candidate_connections), 1) * 100,
        2
    ),
    "%"
)

print("\nRejection reasons:")

display(
    rejected_connections[
        "CONNECTION_REJECTION_REASON"
    ].value_counts().to_frame("Count")
)

CONNECTION VALIDATION RESULTS
--------------------------------------------------
Candidate connections: 511524
Valid connections: 479665
Rejected connections: 31859
Valid connection percentage: 93.77 %

Rejection reasons:


,Count
CONNECTION_REJECTION_REASON,
INVALID_ACTUAL_TURNAROUND; INVALID_SCHEDULED_TURNAROUND,9754
AIRPORT_MISMATCH,7025
INVALID_SCHEDULED_TURNAROUND,6732
AIRPORT_MISMATCH; INVALID_ACTUAL_TURNAROUND; INVALID_SCHEDULED_TURNAROUND,3569
AIRPORT_MISMATCH; INVALID_SCHEDULED_TURNAROUND,2223
AIRPORT_MISMATCH; INVALID_ACTUAL_TIME; INVALID_ACTUAL_TURNAROUND; INVALID_SCHEDULED_TURNAROUND,1259
INVALID_ACTUAL_TURNAROUND,481
AIRPORT_MISMATCH; INVALID_ACTUAL_TIME; INVALID_ACTUAL_TURNAROUND,447
AIRPORT_MISMATCH; INVALID_ACTUAL_TURNAROUND,141


In [9]:
# ================================================================
# 13. AIRCRAFT CONNECTION SUMMARY
# ================================================================

connection_summary = pd.DataFrame({
    "Metric": [
        "Original Rotation Flights",
        "Valid Timestamp Flights",
        "Unique Aircraft",
        "Candidate Connections",
        "Valid Connections",
        "Rejected Connections",
        "Valid Connection Percentage",
        "Average Actual Turnaround (Minutes)",
        "Average Scheduled Turnaround (Minutes)",
        "Average Buffer Consumed (Minutes)"
    ],

    "Value": [
        len(rotation_flights),
        len(phase3_flights),
        phase3_flights["TAIL_NUM"].nunique(),
        len(candidate_connections),
        len(aircraft_connections),
        len(rejected_connections),

        round(
            len(aircraft_connections) /
            max(len(candidate_connections), 1) * 100,
            2
        ),

        round(
            aircraft_connections["ACTUAL_TURNAROUND_MIN"].mean(),
            2
        ),

        round(
            aircraft_connections["SCHEDULED_TURNAROUND_BUFFER_MIN"].mean(),
            2
        ),

        round(
            aircraft_connections["BUFFER_CONSUMED_MIN"].mean(),
            2
        )
    ]
})

display(connection_summary)


# ================================================================
# 14. INSPECT AN AIRCRAFT ROTATION
# ================================================================

def inspect_aircraft_rotation(tail_number):

    aircraft = phase3_flights[
        phase3_flights["TAIL_NUM"] == tail_number
    ].sort_values("DEP_UTC")

    columns = [
        "FLIGHT_ID",
        "FL_DATE",
        "TAIL_NUM",
        "ORIGIN",
        "DEST",
        "DEP_UTC",
        "ARR_UTC",
        "DEP_DELAY",
        "ARR_DELAY"
    ]

    return aircraft[columns].reset_index(drop=True)


if not aircraft_connections.empty:

    sample_tail = aircraft_connections[
        "PREV_TAIL_NUM"
    ].iloc[0]

    print("Sample aircraft:", sample_tail)

    display(inspect_aircraft_rotation(sample_tail))

,Metric,Value
0,Original Rotation Flights,517222.00
1,Valid Timestamp Flights,517222.00
2,Unique Aircraft,5698.00
3,Candidate Connections,511524.00
4,Valid Connections,479665.00
5,Rejected Connections,31859.00
6,Valid Connection Percentage,93.77
7,Average Actual Turnaround (Minutes),211.04
8,Average Scheduled Turnaround (Minutes),206.63
9,Average Buffer Consumed (Minutes),-4.41


Sample aircraft: 188NV


,FLIGHT_ID,FL_DATE,TAIL_NUM,ORIGIN,DEST,DEP_UTC,ARR_UTC,DEP_DELAY,ARR_DELAY
0,FL0,2026-01-01,188NV,TYS,BOS,2026-01-01 14:34:00+00:00,2026-01-01 16:58:00+00:00,-11.0,3.0
1,FL1,2026-01-01,188NV,BOS,TYS,2026-01-01 17:40:00+00:00,2026-01-01 20:07:00+00:00,-5.0,-9.0
2,FL2,2026-01-02,188NV,TYS,EWR,2026-01-02 12:06:00+00:00,2026-01-02 14:00:00+00:00,-9.0,-5.0
3,FL3,2026-01-02,188NV,EWR,TYS,2026-01-02 14:47:00+00:00,2026-01-02 16:48:00+00:00,-8.0,-7.0
4,FL4,2026-01-02,188NV,TYS,PIE,2026-01-02 17:39:00+00:00,2026-01-02 19:22:00+00:00,-6.0,-5.0
5,FL5,2026-01-02,188NV,PIE,TYS,2026-01-02 20:06:00+00:00,2026-01-02 21:53:00+00:00,-11.0,-3.0
6,FL6,2026-01-02,188NV,TYS,SFB,2026-01-02 22:48:00+00:00,2026-01-03 00:31:00+00:00,2.0,10.0
7,FL7,2026-01-02,188NV,SFB,TYS,2026-01-03 01:15:00+00:00,2026-01-03 02:53:00+00:00,4.0,5.0
8,FL8,2026-01-03,188NV,TYS,BOS,2026-01-03 10:48:00+00:00,2026-01-03 12:50:00+00:00,-12.0,-20.0
9,FL9,2026-01-03,188NV,BOS,TYS,2026-01-03 13:53:00+00:00,2026-01-03 16:13:00+00:00,-7.0,-18.0


In [10]:
# ================================================================
# 15. SAVE PHASE 3 OUTPUTS
# ================================================================

os.makedirs(processed_path, exist_ok=True)

phase3_flights.to_csv(
    os.path.join(
        processed_path,
        "phase3_reconstructed_flights.csv"
    ),
    index=False
)

candidate_connections.to_csv(
    os.path.join(
        processed_path,
        "candidate_aircraft_connections.csv"
    ),
    index=False
)

aircraft_connections.to_csv(
    os.path.join(
        processed_path,
        "aircraft_connections.csv"
    ),
    index=False
)

rejected_connections.to_csv(
    os.path.join(
        processed_path,
        "rejected_aircraft_connections.csv"
    ),
    index=False
)

invalid_flights.to_csv(
    os.path.join(
        processed_path,
        "invalid_timestamp_flights.csv"
    ),
    index=False
)

connection_summary.to_csv(
    os.path.join(
        processed_path,
        "aircraft_connection_summary.csv"
    ),
    index=False
)

print("PHASE 3 OUTPUTS SAVED SUCCESSFULLY")

for filename in [
    "phase3_reconstructed_flights.csv",
    "candidate_aircraft_connections.csv",
    "aircraft_connections.csv",
    "rejected_aircraft_connections.csv",
    "invalid_timestamp_flights.csv",
    "aircraft_connection_summary.csv"
]:
    print(filename)

print("\nFinal connection dataset shape:", aircraft_connections.shape)

display(aircraft_connections.head(10))

PHASE 3 OUTPUTS SAVED SUCCESSFULLY
phase3_reconstructed_flights.csv
candidate_aircraft_connections.csv
aircraft_connections.csv
rejected_aircraft_connections.csv
invalid_timestamp_flights.csv
aircraft_connection_summary.csv

Final connection dataset shape: (479665, 35)


,PREV_FLIGHT_ID,PREV_TAIL_NUM,PREV_CARRIER,PREV_FLIGHT_NUM,PREV_FL_DATE,PREV_ORIGIN,PREV_DEST,PREV_DEP_UTC,PREV_ARR_UTC,PREV_CRS_DEP_UTC,...,NEXT_ARR_DELAY,SAME_AIRCRAFT,AIRPORT_CONTINUITY,ACTUAL_TIME_VALID,ACTUAL_TURNAROUND_MIN,SCHEDULED_TURNAROUND_BUFFER_MIN,BUFFER_CONSUMED_MIN,SHORT_TURNAROUND_FLAG,CONNECTION_VALID,CONNECTION_REJECTION_REASON
0,FL0,188NV,G4,1397,2026-01-01,TYS,BOS,2026-01-01 14:34:00+00:00,2026-01-01 16:58:00+00:00,2026-01-01 14:45:00+00:00,...,-9.0,True,True,True,42.0,50.0,8.0,False,True,VALID
1,FL1,188NV,G4,1398,2026-01-01,BOS,TYS,2026-01-01 17:40:00+00:00,2026-01-01 20:07:00+00:00,2026-01-01 17:45:00+00:00,...,-5.0,True,True,True,959.0,959.0,0.0,False,True,VALID
2,FL2,188NV,G4,1396,2026-01-02,TYS,EWR,2026-01-02 12:06:00+00:00,2026-01-02 14:00:00+00:00,2026-01-02 12:15:00+00:00,...,-7.0,True,True,True,47.0,50.0,3.0,False,True,VALID
3,FL3,188NV,G4,1399,2026-01-02,EWR,TYS,2026-01-02 14:47:00+00:00,2026-01-02 16:48:00+00:00,2026-01-02 14:55:00+00:00,...,-5.0,True,True,True,51.0,50.0,-1.0,False,True,VALID
4,FL4,188NV,G4,3030,2026-01-02,TYS,PIE,2026-01-02 17:39:00+00:00,2026-01-02 19:22:00+00:00,2026-01-02 17:45:00+00:00,...,-3.0,True,True,True,44.0,50.0,6.0,False,True,VALID
5,FL5,188NV,G4,1397,2026-01-02,PIE,TYS,2026-01-02 20:06:00+00:00,2026-01-02 21:53:00+00:00,2026-01-02 20:17:00+00:00,...,10.0,True,True,True,55.0,50.0,-5.0,False,True,VALID
6,FL6,188NV,G4,3034,2026-01-02,TYS,SFB,2026-01-02 22:48:00+00:00,2026-01-03 00:31:00+00:00,2026-01-02 22:46:00+00:00,...,5.0,True,True,True,44.0,50.0,6.0,False,True,VALID
7,FL7,188NV,G4,3045,2026-01-02,SFB,TYS,2026-01-03 01:15:00+00:00,2026-01-03 02:53:00+00:00,2026-01-03 01:11:00+00:00,...,-20.0,True,True,True,475.0,492.0,17.0,False,True,VALID
8,FL8,188NV,G4,3040,2026-01-03,TYS,BOS,2026-01-03 10:48:00+00:00,2026-01-03 12:50:00+00:00,2026-01-03 11:00:00+00:00,...,-18.0,True,True,True,63.0,50.0,-13.0,False,True,VALID
9,FL9,188NV,G4,3028,2026-01-03,BOS,TYS,2026-01-03 13:53:00+00:00,2026-01-03 16:13:00+00:00,2026-01-03 14:00:00+00:00,...,-19.0,True,True,True,62.0,50.0,-12.0,False,True,VALID
